# GOV-01 V2-E2: Frozen MobileNetV2 for Seven-Class Road Conditions

This controlled experiment compares frozen ImageNet-pretrained MobileNetV2 with the completed V2-E1 compact CNN. It uses only V2 `train` and `validation` images. It must not load, inspect, or evaluate `protected_test`.

The V2-E1 validation benchmark is Macro F1 `0.6796` and accuracy `0.6984`. V2-E2 is better only if validation evidence improves without unacceptable per-class regressions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile

ARCHIVE_PATH = Path('/content/drive/MyDrive/GOV-01/v2/multiclass_final.zip')
DATA_DIR = Path('/content/data/processed/v2/multiclass_final')
OUTPUT_DIR = Path('/content/v2_e2_output')
PERSISTENT_DIR = Path('/content/drive/MyDrive/GOV-01/v2_e2_checkpoint')

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f'Upload the persistent dataset ZIP to Google Drive first: {ARCHIVE_PATH}')
if not DATA_DIR.is_dir():
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(DATA_DIR.parent)
    print('Extracted from Google Drive:', DATA_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)
print('Training data:', DATA_DIR)
print('Temporary Colab output folder:', OUTPUT_DIR)
print('Persistent checkpoint folder:', PERSISTENT_DIR)
print('The protected-test folder is deliberately not loaded in this notebook.')

In [ ]:
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ['crack', 'manhole_cover', 'normal_asphalt', 'pothole', 'repaired_road', 'speed_bump', 'unpaved_road']
V2_E1_MACRO_F1 = 0.6795904943261485
V2_E1_ACCURACY = 0.6984318455971049

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print('TensorFlow:', tf.__version__)
print('Visible GPUs:', tf.config.list_physical_devices('GPU'))
print('Class order:', CLASS_NAMES)

In [ ]:
def count_images(split_name):
    result = {}
    for class_name in CLASS_NAMES:
        class_dir = DATA_DIR / split_name / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f'Missing class folder: {class_dir}')
        result[class_name] = len([path for path in class_dir.iterdir() if path.is_file()])
    return result

train_counts = count_images('train')
validation_counts = count_images('validation')
print('Train:', train_counts, 'total=', sum(train_counts.values()))
print('Validation:', validation_counts, 'total=', sum(validation_counts.values()))
assert sum(train_counts.values()) == 9452
assert sum(validation_counts.values()) == 1658

In [ ]:
# Only train and validation are loaded. Validation receives no augmentation.
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train', class_names=CLASS_NAMES, label_mode='int', image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED,
)
validation_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'validation', class_names=CLASS_NAMES, label_mode='int', image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=False,
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)

In [ ]:
# V2-E2: the pretrained feature extractor is frozen. Only the new seven-class head learns.
# The first run may download ImageNet weights. If that download fails, stop and show the error.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,), include_top=False, weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax', name='road_condition')(x)
model = tf.keras.Model(inputs, outputs, name='v2_e2_frozen_mobilenetv2')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)
model.summary()

In [ ]:
# These files are stored in Google Drive so they survive a Colab disconnect.
checkpoint_path = PERSISTENT_DIR / 'v2_e2_frozen_mobilenetv2_best.keras'
history_path = PERSISTENT_DIR / 'v2_e2_training_history.csv'
summary_path = PERSISTENT_DIR / 'v2_e2_training_summary.json'
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.CSVLogger(history_path, append=False),
]

start_time = time.time()
history = model.fit(train_ds, validation_data=validation_ds, epochs=20, callbacks=callbacks, verbose=1)
training_seconds = time.time() - start_time
summary_path.write_text(json.dumps({
    'training_seconds': training_seconds,
    'epochs_completed': len(history.history['loss']),
}, indent=2) + '\n')
print(f'Training time: {training_seconds:.1f} seconds')
print('Best checkpoint and training history saved in Google Drive.')

In [ ]:
# Always reload the saved best checkpoint. This also works after a Colab reconnect:
# rerun setup cells 1–4, then run this cell. Never rerun training only to make results.
checkpoint_path = PERSISTENT_DIR / 'v2_e2_frozen_mobilenetv2_best.keras'
history_path = PERSISTENT_DIR / 'v2_e2_training_history.csv'
summary_path = PERSISTENT_DIR / 'v2_e2_training_summary.json'

if not checkpoint_path.is_file():
    raise FileNotFoundError(f'Best checkpoint not found: {checkpoint_path}')
if not history_path.is_file():
    raise FileNotFoundError(f'Training history not found: {history_path}')

best_model = tf.keras.models.load_model(checkpoint_path)
history_rows = list(__import__('csv').DictReader(history_path.read_text().splitlines()))
history_values = {
    key: [float(row[key]) for row in history_rows]
    for key in ['loss', 'val_loss']
}
training_summary = json.loads(summary_path.read_text()) if summary_path.is_file() else {}

y_validation = np.concatenate([labels.numpy() for _, labels in validation_ds])
probabilities = best_model.predict(validation_ds, verbose=0)
predictions = probabilities.argmax(axis=1)

e2_metrics = {
    'run_name': 'v2_e2_frozen_mobilenetv2',
    'validation_accuracy': float(accuracy_score(y_validation, predictions)),
    'validation_macro_f1': float(f1_score(y_validation, predictions, average='macro', zero_division=0)),
    'epochs_completed': len(history_rows),
    'training_seconds': training_summary.get('training_seconds'),
    'class_weight': 'none',
    'base_model': 'MobileNetV2 ImageNet weights, frozen',
    'classification_report': classification_report(y_validation, predictions, target_names=CLASS_NAMES, output_dict=True, zero_division=0),
}
comparison = {
    'v2_e1_compact_cnn_macro_f1': V2_E1_MACRO_F1,
    'v2_e2_frozen_mobilenetv2_macro_f1': e2_metrics['validation_macro_f1'],
    'macro_f1_difference_e2_minus_e1': e2_metrics['validation_macro_f1'] - V2_E1_MACRO_F1,
    'v2_e1_compact_cnn_accuracy': V2_E1_ACCURACY,
    'v2_e2_frozen_mobilenetv2_accuracy': e2_metrics['validation_accuracy'],
}
(OUTPUT_DIR / 'v2_e2_validation_metrics.json').write_text(json.dumps(e2_metrics, indent=2) + '\n')
(OUTPUT_DIR / 'v2_e2_comparison.json').write_text(json.dumps(comparison, indent=2) + '\n')
print(json.dumps(comparison, indent=2))

ConfusionMatrixDisplay.from_predictions(y_validation, predictions, display_labels=CLASS_NAMES, xticks_rotation=45)
plt.title('V2-E2 frozen MobileNetV2 validation confusion matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v2_e2_validation_confusion_matrix.png', dpi=150)
plt.show()

plt.plot(history_values['loss'], label='train loss')
plt.plot(history_values['val_loss'], label='validation loss')
plt.title('V2-E2 frozen MobileNetV2 loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v2_e2_learning_curve.png', dpi=150)
plt.show()

# Include a copy in the downloadable results ZIP. It remains ignored by Git.
import shutil
shutil.copy2(checkpoint_path, OUTPUT_DIR / checkpoint_path.name)

print('V2-E2 validation evidence saved locally. Do not load protected_test.')

In [ ]:
# Run this only after you have reviewed the V2-E2 validation output.
# It downloads metrics, plots, comparison, and the model checkpoint before Colab resets.
import shutil
from google.colab import files

archive_path = shutil.make_archive('/content/v2_e2_output', 'zip', OUTPUT_DIR)
files.download(archive_path)